In [9]:
"""PneumoScan - model training, confidence gate and gated-pipeline evaluation (v2).

Single-file version of the notebook. Cells are marked with # %%.
Convert to a notebook:  pip install jupytext && jupytext --to notebook PneumoScan_training_v2.py

Smoke test without GPU or dataset downloads (tiny models, tiny images):
    PNEUMOSCAN_SMOKE=1 PNEUMOSCAN_DATA=<folder with train/val/test> python PneumoScan_training_v2.py
"""

'PneumoScan - model training, confidence gate and gated-pipeline evaluation (v2).\n\nSingle-file version of the notebook. Cells are marked with # %%.\nConvert to a notebook:  pip install jupytext && jupytext --to notebook PneumoScan_training_v2.py\n\nSmoke test without GPU or dataset downloads (tiny models, tiny images):\n    PNEUMOSCAN_SMOKE=1 PNEUMOSCAN_DATA=<folder with train/val/test> python PneumoScan_training_v2.py\n'

# PneumoScan — training, confidence gate and gated-pipeline evaluation (revised)

Trains and compares **seven models** on the Kaggle Chest X-Ray Images (Pneumonia) dataset,
fits the confidence gate, and then evaluates the **whole gated pipeline**, not the classifier alone.

**Classical baselines:** logistic regression, RBF-SVM, random forest (HOG + downsampled pixels)
**Deep models:** small CNN from scratch, ResNet50, DenseNet121, EfficientNetB0 (two-stage fine-tuning)

### What changed from v1 (response to supervisor review)
| Issue | notebook 1 | revised |
|---|---|---|
| Unit of splitting | patient (train/val), but never checked against test | patient, **checked against the original test folder too**; overlapping patients are excluded |
| One validation set did everything | early stopping, model selection, temperature, both thresholds | **four disjoint patient-level sets**: train, selection, calibration, test |
| Reproducibility | `splits.csv` only | **`manifest.csv`** (file, MD5, patient, label, split, size) + SHA-256 of the manifest, seed, library versions |
| Duplicate handling | first copy kept, order depended on the file system | deterministic: the test-folder copy is kept, otherwise the first path alphabetically |
| Decision threshold | sensitivity-target threshold (≈0.975) re-used as the confidence threshold | **decision threshold fixed at 0.5**; confidence threshold and distance threshold defined separately, both fitted on the calibration set |
| Feature statistics | computed on augmented training images | computed on **un-augmented** training images |
| Uncertainty | image-level bootstrap | **patient-level (cluster) bootstrap**, Wilson intervals for proportions |
| Gate evaluation | accepted-case sensitivity/specificity | retained coverage, per-class outcomes, per-class errors with intervals, false-clearance rate, coverage–risk table, every accepted false negative listed |
| Out-of-distribution | not evaluated | degraded, non-radiographic and (optional) external folders pushed through the frozen gate |

## 1. Setup

In [10]:
import os, re, sys, json, copy, time, hashlib, random, platform
from datetime import datetime, timezone
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, roc_curve

SMOKE = os.environ.get("PNEUMOSCAN_SMOKE") == "1"      # tiny run for testing the code path only

SEED = 42
IMG_SIZE = 64 if SMOKE else 224
BATCH = 32
EPOCHS_HEAD, EPOCHS_FT, PATIENCE = (1, 1, 1) if SMOKE else (4, 8, 3)
NUM_WORKERS = 0 if SMOKE else 2

# ---- split design (patient level) ----
FRACTIONS = {"train": 0.70, "sel": 0.15, "cal": 0.15}   # of the train+val pool; test = original test folder

# ---- gate design: every parameter is defined here and fitted on the calibration set only ----
DECISION_THR = 0.5      # class decision on the calibrated probability. Fixed in advance, never tuned.
RETAIN_DIST = 0.95      # distance threshold keeps this share of in-distribution calibration images
COVERAGE_CONF = 0.80    # confidence threshold keeps this share of the remaining calibration images

def find_data_root():
    """Kaggle mounts this dataset at different depths. Find the folder that
    directly contains train/, val/ and test/."""
    env = os.environ.get("PNEUMOSCAN_DATA")
    if env:
        return Path(env)
    for root, dirs, _ in os.walk("/kaggle/input"):
        if {"train", "val", "test"} <= set(dirs):
            return Path(root)
    return Path("/kaggle/input/chest-xray-pneumonia/chest_xray")

DATA_ROOT = find_data_root()
OUT = Path(os.environ.get("PNEUMOSCAN_OUT", "/kaggle/working/pneumoscan"))
OUT.mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| smoke test:", SMOKE)
print("data root exists:", DATA_ROOT.exists(), "|", DATA_ROOT)

device: cuda | smoke test: False
data root exists: True | /kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray


In [11]:
def confusion(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    return tp, tn, fp, fn

def metrics_at(y_true, probs, threshold=DECISION_THR):
    y_pred = (np.asarray(probs) >= threshold).astype(int)
    tp, tn, fp, fn = confusion(y_true, y_pred)
    sens = tp / (tp + fn) if tp + fn else float("nan")
    spec = tn / (tn + fp) if tn + fp else float("nan")
    prec = tp / (tp + fp) if tp + fp else float("nan")
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else float("nan")
    try:
        auc = roc_auc_score(y_true, probs)
    except ValueError:
        auc = float("nan")
    return {"n": int(len(y_true)), "threshold": float(threshold),
            "sensitivity": sens, "specificity": spec, "precision": prec, "f1": f1,
            "accuracy": (tp + tn) / max(1, tp + tn + fp + fn), "auc": auc,
            "tp": tp, "tn": tn, "fp": fp, "fn": fn}

def wilson(k, n, z=1.96):
    """Wilson score interval for a proportion k/n."""
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return (float(centre - half), float(centre + half))

def cluster_bootstrap(groups, stat_fn, n_boot=1000, seed=SEED):
    """Patient-level bootstrap: resample whole patients, so several images of one patient are
    never treated as independent. stat_fn(idx) -> dict of statistics for the resampled rows."""
    codes, _ = pd.factorize(np.asarray(groups))
    members = [np.where(codes == k)[0] for k in range(codes.max() + 1)]
    rng = np.random.default_rng(seed)
    draws = []
    for _ in range(n_boot):
        pick = rng.integers(0, len(members), len(members))
        draws.append(stat_fn(np.concatenate([members[k] for k in pick])))
    return pd.DataFrame(draws)

def ci_of(draws, col):
    v = draws[col].to_numpy(dtype=float)
    v = v[~np.isnan(v)]
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if len(v) else (np.nan, np.nan)

def expected_calibration_error(y_true, probs, bins=15):
    y_true, probs = np.asarray(y_true), np.asarray(probs)
    conf = np.where(probs >= 0.5, probs, 1 - probs)
    correct = ((probs >= 0.5).astype(int) == y_true).astype(float)
    edges, ece = np.linspace(0, 1, bins + 1), 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    return np.where(x >= 0, 1.0 / (1.0 + np.exp(-np.abs(x))), np.exp(-np.abs(x)) / (1.0 + np.exp(-np.abs(x))))

def save_json(obj, name):
    with open(OUT / name, "w") as f:
        json.dump(obj, f, indent=2, default=float)
    print("wrote", OUT / name)

## 2. Data audit, manifest and patient-level splits

Filenames carry the patient: `person53_bacteria_255.jpeg` → `person53`, and normal images
`IM-0115-0001.jpeg` → `IM-0115`. **The unit of splitting is the patient.**

Four disjoint sets, each with one job:

| Set | Source | Used for | Never used for |
|---|---|---|---|
| `train` (70%) | train+val folders | fitting weights; class means and covariance for the distance score | any threshold |
| `sel` (15%) | train+val folders | early stopping, choosing which model to deploy | calibration, thresholds, reporting |
| `cal` (15%) | train+val folders | temperature, distance threshold, confidence threshold | model selection, reporting |
| `test` | original test folder | final reporting only | everything else |

In [12]:
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}

def patient_id(path):
    name = Path(path).name
    m = re.search(r"(person\d+)", name)
    if m: return m.group(1)
    m = re.search(r"(IM-\d+)", name)
    if m: return m.group(1)
    return Path(path).stem

def file_hash(path, blocksize=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while chunk := f.read(blocksize):
            h.update(chunk)
    return h.hexdigest()

rows = []
for folder in ("train", "val", "test"):
    d = DATA_ROOT / folder
    if not d.exists():
        print("missing:", d)
        continue
    for p in d.rglob("*"):
        if p.suffix.lower() not in {".jpeg", ".jpg", ".png"}:
            continue
        parts = {q.upper() for q in p.parts}
        if "NORMAL" in parts:
            label = 0
        elif "PNEUMONIA" in parts:
            label = 1
        else:
            continue
        rows.append((str(p), p.relative_to(DATA_ROOT).as_posix(), label, patient_id(p), folder))

df = pd.DataFrame(rows, columns=["path", "file", "label", "patient_id", "original_folder"])
df = df.sort_values("file").reset_index(drop=True)          # deterministic order everywhere downstream
df["file_md5"] = df["path"].map(file_hash)
def read_size(p):
    """(width, height), or (None, None) if the file cannot be decoded as an image."""
    try:
        with Image.open(p) as im:
            im.load()                       # forces a full decode, so truncated files are caught here
            return im.size
    except Exception:
        return (None, None)

sizes = [read_size(p) for p in df["path"]]
df["width"] = [s_[0] for s_ in sizes]; df["height"] = [s_[1] for s_ in sizes]
df["readable"] = df["width"].notna()
n_found = len(df)
n_unreadable = int((~df.readable).sum())
print(f"files that cannot be decoded as images: {n_unreadable} (excluded and listed in the manifest)")
print(f"found {n_found} images, {df.patient_id.nunique()} patient groups")
print(df.groupby(["original_folder", "label"]).size().unstack(fill_value=0))

files that cannot be decoded as images: 0 (excluded and listed in the manifest)
found 5856 images, 2790 patient groups
label               0     1
original_folder            
test              234   390
train            1341  3875
val                 8     8


In [13]:
# Duplicates. Deterministic rule: keep the copy in the test folder if there is one (so the test
# set is never silently shrunk to favour training), otherwise the first path alphabetically.
df["status"] = np.where(df.readable, "kept", "excluded_unreadable")
df["duplicate_of"] = ""
label_conflicts = 0
for h, g in df[df.status == "kept"].groupby("file_md5"):
    if len(g) < 2:
        continue
    in_test = g[g.original_folder == "test"]
    keep = (in_test if len(in_test) else g).index[0]
    if g.label.nunique() > 1:
        label_conflicts += 1
    for i in g.index:
        if i != keep:
            df.loc[i, "status"] = "removed_duplicate"
            df.loc[i, "duplicate_of"] = df.loc[keep, "file"]
n_dupes = int((df.status == "removed_duplicate").sum())
print(f"duplicate files removed: {n_dupes}   (duplicate groups with conflicting labels: {label_conflicts})")

# Patients present in the original test folder must not be present in training data either.
kept = df[df.status == "kept"]
test_patients = set(kept[kept.original_folder == "test"].patient_id)
# A training image that is an exact copy of a test image links its patient to the test set, even if the
# file names carry different patient identifiers.
linked = df[(df.status == "removed_duplicate") & (df.original_folder != "test") & df.duplicate_of.str.startswith("test/")]
test_patients |= set(linked.patient_id)
print(f"patients linked to the test set through duplicate images: {linked.patient_id.nunique()}")
overlap_mask = (df.status == "kept") & (df.original_folder != "test") & df.patient_id.isin(test_patients)
df.loc[overlap_mask, "status"] = "excluded_patient_overlap_with_test"
n_overlap = int(overlap_mask.sum())
print(f"images excluded because their patient also appears in the test folder: {n_overlap}")

duplicate files removed: 32   (duplicate groups with conflicting labels: 0)
patients linked to the test set through duplicate images: 0
images excluded because their patient also appears in the test folder: 547


In [14]:
def patient_split(pool, fractions, seed=SEED):
    """Stratified (by the patient's majority label) patient-level split into len(fractions) sets."""
    pat_label = pool.groupby("patient_id")["label"].agg(lambda s: int(s.mean() >= 0.5))
    rng = np.random.default_rng(seed)
    names = list(fractions)
    cuts = np.cumsum([fractions[k] for k in names])
    assign = {}
    for cls in (0, 1):
        ids = sorted(pat_label.index[pat_label == cls])
        rng.shuffle(ids)
        bounds = [int(round(c * len(ids))) for c in cuts]; bounds[-1] = len(ids)
        start = 0
        for name, b in zip(names, bounds):
            for pid in ids[start:b]:
                assign[pid] = name
            start = b
    return pool["patient_id"].map(assign)

pool = df[(df.status == "kept") & (df.original_folder != "test")].copy()
pool["split"] = patient_split(pool, FRACTIONS)
df["split"] = df["status"].where(df.status != "kept", "")
df.loc[pool.index, "split"] = pool["split"]
df.loc[(df.status == "kept") & (df.original_folder == "test"), "split"] = "test"

used = df[df.split.isin(["train", "sel", "cal", "test"])]
names = ["train", "sel", "cal", "test"]
for i, a in enumerate(names):
    for b in names[i + 1:]:
        shared = set(used[used.split == a].patient_id) & set(used[used.split == b].patient_id)
        assert not shared, f"patient leakage between {a} and {b}: {sorted(shared)[:5]}"
    assert used[used.split == a].label.nunique() == 2, f"{a} is missing a class"
print("\nno patient appears in more than one of train / sel / cal / test\n")

splits = used.reset_index(drop=True)
counts = splits.groupby(["split", "label"]).size().unstack(fill_value=0)
counts.columns = ["normal", "pneumonia"]
counts = counts.loc[names]
counts["images"] = counts.sum(axis=1)
counts["patients"] = splits.groupby("split").patient_id.nunique()
print(counts)


no patient appears in more than one of train / sel / cal / test

       normal  pneumonia  images  patients
split                                     
train     887       2426    3313      1670
sel       195        535     730       358
cal       172        444     616       358
test      231        387     618       403


In [15]:
# The manifest: every file found, with what happened to it. This is the reproducibility record.
manifest = df[["file", "file_md5", "patient_id", "label", "original_folder", "split",
               "duplicate_of", "width", "height"]].copy()
manifest["label"] = manifest["label"].map({0: "NORMAL", 1: "PNEUMONIA"})
manifest.to_csv(OUT / "manifest.csv", index=False)
manifest_sha256 = hashlib.sha256((OUT / "manifest.csv").read_bytes()).hexdigest()
print("manifest.csv rows:", len(manifest), "| sha256:", manifest_sha256[:16], "...")

env = {"date_utc": datetime.now(timezone.utc).isoformat(), "python": platform.python_version(),
       "torch": torch.__version__, "numpy": np.__version__, "pandas": pd.__version__,
       "seed": SEED, "fractions": FRACTIONS, "decision_threshold": DECISION_THR,
       "retain_dist": RETAIN_DIST, "coverage_conf": COVERAGE_CONF, "smoke_test": SMOKE}
save_json(env, "environment.json")

save_json({"files_found": n_found,
           "unreadable_files_excluded": n_unreadable,
           "valid_images_after_decoding": n_found - n_unreadable,
           "duplicates_removed": n_dupes,
           "duplicate_groups_with_conflicting_labels": label_conflicts,
           "excluded_patient_overlap_with_test": n_overlap,
           "images_used": int(len(splits)),
           "unique_patients_used": int(splits.patient_id.nunique()),
           "unit_of_splitting": "patient",
           "split_counts": counts.to_dict(),
           "width": {"min": int(df.width.min()), "max": int(df.width.max())},
           "height": {"min": int(df.height.min()), "max": int(df.height.max())},
           "files_by_folder_and_label": {f"{k[0]}/{k[1]}": int(v) for k, v in df.groupby(["original_folder", "label"]).size().items()},
           "patient_ids_by_split_and_class": {f"{k[0]}/{k[1]}": int(v) for k, v in splits.groupby(["split", "label"]).patient_id.nunique().items()},
           "patient_identifier_caveat": "pneumonia identifiers (personN) are patient-level; normal identifiers (IM-xxxx) are a weaker proxy and may group images of different patients or split one patient",
           "manifest_sha256": manifest_sha256}, "audit_report.json")

manifest.csv rows: 5856 | sha256: 3118e967eeefe123 ...
wrote /kaggle/working/pneumoscan/environment.json
wrote /kaggle/working/pneumoscan/audit_report.json


In [16]:
ax = counts[["normal", "pneumonia"]].plot(kind="bar", figsize=(6.4, 3.6), color=["#7FA7D9", "#2F4B7C"], rot=0)
ax.set_ylabel("images"); ax.set_xlabel(""); ax.set_title("Class distribution after patient-level split")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.savefig(OUT / "class_distribution.png", dpi=200); plt.show()

## 3. Classical baselines

Three models on HOG features plus downsampled pixels. These set the floor: every claim about
the deep models is stated relative to what a classical model already achieves on the same split.
All models are evaluated at the same fixed decision threshold (0.5).

In [17]:
from skimage.feature import hog
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SMALL = 64

def features(path):
    with Image.open(path) as im:
        g = im.convert("L").resize((SMALL, SMALL))
    a = np.asarray(g, dtype=np.float32) / 255.0
    hg = hog(a, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(2, 2),
             feature_vector=True)
    return np.concatenate([hg, a[::2, ::2].ravel()])

def build_xy(split):
    sub = splits[splits.split == split]
    X = np.stack([features(p) for p in sub.path])
    return X, sub.label.to_numpy()

t0 = time.time()
Xtr, ytr = build_xy("train"); Xsel, ysel = build_xy("sel"); Xte, yte = build_xy("test")
scaler = StandardScaler().fit(Xtr)
Xtr, Xsel, Xte = scaler.transform(Xtr), scaler.transform(Xsel), scaler.transform(Xte)
print(f"features ready in {(time.time()-t0)/60:.1f} min — {Xtr.shape[1]} dims")

features ready in 0.9 min — 2788 dims


In [18]:
classical_models = {
    "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced",
                                              C=1.0, random_state=SEED),
    "svm_rbf": SVC(kernel="rbf", C=10.0, gamma="scale", probability=True,
                   class_weight="balanced", random_state=SEED),
    "random_forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=2,
                                            class_weight="balanced_subsample",
                                            n_jobs=-1, random_state=SEED),
}

classical_results, classical_probs = {}, {}
for name, clf in classical_models.items():
    print(f"training {name} ...")
    clf.fit(Xtr, ytr)
    psel, pte = clf.predict_proba(Xsel)[:, 1], clf.predict_proba(Xte)[:, 1]
    classical_results[name] = {"sel": metrics_at(ysel, psel), "test": metrics_at(yte, pte)}
    classical_probs[name] = pte
    m = classical_results[name]["test"]
    print(f"   test  sens={m['sensitivity']:.3f}  spec={m['specificity']:.3f}  auc={m['auc']:.3f}")

np.save(OUT / "y_test.npy", yte)
save_json(classical_results, "results_classical.json")

training logistic_regression ...
   test  sens=0.992  spec=0.476  auc=0.938
training svm_rbf ...
   test  sens=0.995  spec=0.433  auc=0.940
training random_forest ...
   test  sens=0.992  spec=0.377  auc=0.949
wrote /kaggle/working/pneumoscan/results_classical.json


## 4. Deep models

A small CNN from scratch as the deep baseline, then three ImageNet backbones fine-tuned in two
stages: head only at 1e-3, then the whole network at 1e-4. AdamW, weight decay 1e-4, BCE with
`pos_weight`. **Early stopping uses the selection set only.**

No horizontal flipping — it mirrors anatomy and produces images that are not valid radiographs.


In [19]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class CXR(Dataset):
    def __init__(self, frame, tf):
        self.paths, self.labels, self.tf = frame.path.tolist(), frame.label.tolist(), tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return self.tf(im.convert("RGB")), self.labels[i]

def make_loader(name, tf, shuffle=False):
    return DataLoader(CXR(splits[splits.split == name], tf), batch_size=BATCH, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEV == "cuda"))

loaders = {
    "train": make_loader("train", train_tf, shuffle=True),
    "train_eval": make_loader("train", eval_tf),     # un-augmented, for the distance statistics
    "sel": make_loader("sel", eval_tf),
    "cal": make_loader("cal", eval_tf),
    "test": make_loader("test", eval_tf),
}
ytr_deep = splits[splits.split == "train"].label.to_numpy()
pos_weight = torch.tensor([(ytr_deep == 0).sum() / max(1, (ytr_deep == 1).sum())],
                          dtype=torch.float32, device=DEV)
print("class imbalance handled with pos_weight =", float(pos_weight))

class imbalance handled with pos_weight = 0.3656224310398102


In [20]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        blocks, c_in = [], 3
        for c_out in ((8, 16, 16, 16) if SMOKE else (32, 64, 128, 128)):
            blocks += [nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c_out
        self.features = nn.Sequential(*blocks, nn.AdaptiveAvgPool2d(1))
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(c_in, 1))
    def forward(self, x):
        return self.classifier(self.features(x))
    def embed(self, x):
        return torch.flatten(self.features(x), 1)

def build(name):
    if name == "small_cnn" or SMOKE:                 # smoke test: tiny model for every name
        m = SmallCNN()
        return m, list(m.features.parameters()), list(m.classifier.parameters())
    if name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        head = nn.Linear(m.fc.in_features, 1); m.fc = head
    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        head = nn.Linear(m.classifier.in_features, 1); m.classifier = head
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        head = nn.Linear(m.classifier[1].in_features, 1); m.classifier[1] = head
    else:
        raise ValueError(name)
    head_ids = {id(p) for p in head.parameters()}
    backbone = [p for p in m.parameters() if id(p) not in head_ids]
    return m, backbone, list(head.parameters())

def embed(model, name, x):
    """Penultimate-layer features — the distance score in section 5 is built on these."""
    if name == "small_cnn" or SMOKE:
        return model.embed(x)
    if name == "resnet50":
        return torch.flatten(nn.Sequential(*list(model.children())[:-1])(x), 1)
    if name == "densenet121":
        f = torch.relu(model.features(x))
        return torch.flatten(F.adaptive_avg_pool2d(f, 1), 1)
    return torch.flatten(model.avgpool(model.features(x)), 1)

@torch.no_grad()
def infer(model, name, loader, with_features=False):
    model.eval()
    logits, labels, feats = [], [], []
    for x, y in loader:
        x = x.to(DEV, non_blocking=True)
        logits.append(model(x).squeeze(1).cpu().numpy())
        labels.append(np.asarray(y))
        if with_features:
            feats.append(embed(model, name, x).cpu().numpy())
    out = [np.concatenate(logits), np.concatenate(labels)]
    if with_features:
        out.append(np.concatenate(feats))
    return out

In [21]:
def run_epochs(model, name, params, lr, epochs, best_state, best_loss, patience):
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    cpu_crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight.cpu())
    for ep in range(epochs):
        model.train(); total = 0.0
        for x, y in loaders["train"]:
            x = x.to(DEV, non_blocking=True); y = y.float().unsqueeze(1).to(DEV)
            opt.zero_grad(); loss = crit(model(x), y); loss.backward(); opt.step()
            total += loss.item() * x.size(0)
        slog, slab = infer(model, name, loaders["sel"])           # selection set, not calibration
        sloss = float(cpu_crit(torch.tensor(slog).unsqueeze(1),
                               torch.tensor(slab).float().unsqueeze(1)))
        print(f"      epoch {ep+1}: train {total/len(loaders['train'].dataset):.4f}  sel {sloss:.4f}")
        if sloss < best_loss:
            best_loss, best_state, patience = sloss, copy.deepcopy(model.state_dict()), PATIENCE
        else:
            patience -= 1
            if patience <= 0:
                print("      early stopping"); break
    return best_state, best_loss, patience

DEEP = ["small_cnn", "resnet50", "densenet121", "efficientnet_b0"]
training_summary = {}

for name in DEEP:
    print(f"\n=== {name} ===")
    t0 = time.time()
    set_seed()
    model, backbone, head = build(name); model.to(DEV)
    best_state, best_loss, patience = None, float("inf"), PATIENCE

    if name == "small_cnn":
        best_state, best_loss, patience = run_epochs(
            model, name, list(model.parameters()), 1e-3,
            EPOCHS_HEAD + EPOCHS_FT, best_state, best_loss, patience)
    else:
        print("   stage 1: head only")
        for p in backbone: p.requires_grad = False
        best_state, best_loss, patience = run_epochs(
            model, name, head, 1e-3, EPOCHS_HEAD, best_state, best_loss, patience)
        print("   stage 2: fine-tuning")
        for p in backbone: p.requires_grad = True
        best_state, best_loss, patience = run_epochs(
            model, name, list(model.parameters()), 1e-4, EPOCHS_FT,
            best_state, best_loss, PATIENCE)

    model.load_state_dict(best_state)
    torch.save(best_state, OUT / f"{name}.pt")

    slog, slab, sfeat = infer(model, name, loaders["sel"], True)
    clog, clab, cfeat = infer(model, name, loaders["cal"], True)
    tlog, tlab, tfeat = infer(model, name, loaders["test"], True)
    _, trlab, trfeat = infer(model, name, loaders["train_eval"], True)
    np.savez(OUT / f"outputs_{name}.npz",
             sel_logits=slog, sel_labels=slab, sel_feats=sfeat,
             cal_logits=clog, cal_labels=clab, cal_feats=cfeat,
             test_logits=tlog, test_labels=tlab, test_feats=tfeat,
             train_feats=trfeat, train_labels=trlab)

    training_summary[name] = {
        "params_millions": round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
        "best_selection_loss": best_loss,
        "minutes": round((time.time() - t0) / 60, 1)}
    print(f"   {training_summary[name]['minutes']} min, "
          f"{training_summary[name]['params_millions']}M params")
    del model; torch.cuda.empty_cache()

save_json(training_summary, "training_summary.json")


=== small_cnn ===
      epoch 1: train 0.2309  sel 0.1825
      epoch 2: train 0.1806  sel 0.1348
      epoch 3: train 0.1544  sel 0.4674
      epoch 4: train 0.1326  sel 0.1867
      epoch 5: train 0.1291  sel 0.1518
      early stopping
   4.4 min, 0.24M params

=== resnet50 ===
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 229MB/s]


   stage 1: head only
      epoch 1: train 0.2434  sel 0.1961
      epoch 2: train 0.1663  sel 0.1550
      epoch 3: train 0.1519  sel 0.1399
      epoch 4: train 0.1387  sel 0.1292
   stage 2: fine-tuning
      epoch 1: train 0.0695  sel 0.0457
      epoch 2: train 0.0448  sel 0.0753
      epoch 3: train 0.0367  sel 0.0241
      epoch 4: train 0.0230  sel 0.0353
      epoch 5: train 0.0223  sel 0.0323
      epoch 6: train 0.0257  sel 0.0298
      early stopping
   8.5 min, 23.51M params

=== densenet121 ===
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 246MB/s]


   stage 1: head only
      epoch 1: train 0.2370  sel 0.1855
      epoch 2: train 0.1534  sel 0.1490
      epoch 3: train 0.1328  sel 0.1262
      epoch 4: train 0.1173  sel 0.1194
   stage 2: fine-tuning
      epoch 1: train 0.0628  sel 0.0472
      epoch 2: train 0.0326  sel 0.0509
      epoch 3: train 0.0390  sel 0.0863
      epoch 4: train 0.0208  sel 0.0281
      epoch 5: train 0.0158  sel 0.0233
      epoch 6: train 0.0143  sel 0.0335
      epoch 7: train 0.0154  sel 0.0376
      epoch 8: train 0.0143  sel 0.0311
      early stopping
   10.0 min, 6.95M params

=== efficientnet_b0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 217MB/s]

   stage 1: head only


      epoch 1: train 0.2348  sel 0.1580
      epoch 2: train 0.1540  sel 0.1256
      epoch 3: train 0.1367  sel 0.1235
      epoch 4: train 0.1300  sel 0.1135
   stage 2: fine-tuning
      epoch 1: train 0.0751  sel 0.0447
      epoch 2: train 0.0460  sel 0.0624
      epoch 3: train 0.0344  sel 0.0313
      epoch 4: train 0.0286  sel 0.0330
      epoch 5: train 0.0281  sel 0.0376
      epoch 6: train 0.0266  sel 0.0411
      early stopping
   8.0 min, 4.01M params
wrote /kaggle/working/pneumoscan/training_summary.json


## 5. Calibration and the confidence gate

**Model selection** uses the selection set. Everything below is then fitted on the **calibration set** only and frozen.

The gate has four parameters. Each is defined here once, and the same definitions are used by the web app.

| Symbol | Name | Definition | Fitted on |
|---|---|---|---|
| *T* | temperature | one scalar; calibrated probability p = sigmoid(logit / T); minimises log-loss | calibration set |
| *t* | decision threshold | **0.5**; label is pneumonia if p ≥ t. Fixed in advance, not tuned | none |
| *c(x)* | confidence | max(p, 1 − p): the calibrated probability of the predicted class | per image |
| *d(x)* | distance score | smallest squared Mahalanobis distance from the penultimate-layer features to either class mean (shared covariance, statistics from un-augmented training images) | none |
| *τ_d* | distance threshold | the 95th percentile of *d* on the calibration set (keeps 95% of in-distribution images) | calibration set |
| *τ_c* | confidence threshold | the smallest *c* among the 80% most confident calibration images that pass the distance test | calibration set |

**Decision rule.** REJECTED if *d* > *τ_d*; otherwise ESCALATED if *c* < *τ_c*; otherwise FLAGGED_URGENT if p ≥ *t*, else CLEARED.

**Confidence score vs validated probability.** *c* is only a *calibrated* score. It becomes a *validated* probability
to the extent that calibration holds on data that were not used to fit *T*. That is why the reliability diagram and ECE are
reported on the **test set** as well as the calibration set.

In [22]:
sel_auc = {}
for name in DEEP:
    d = np.load(OUT / f"outputs_{name}.npz")
    sel_auc[name] = metrics_at(d["sel_labels"], sigmoid(d["sel_logits"]))["auc"]
    print(f"{name:18s} selection-set AUC {sel_auc[name]:.4f}")

best = max(sel_auc, key=sel_auc.get)
print(f"\nselected model: {best}")

d = np.load(OUT / f"outputs_{best}.npz")
cal_logits, cal_labels, cal_feats = d["cal_logits"], d["cal_labels"], d["cal_feats"]
test_logits, test_labels, test_feats = d["test_logits"], d["test_labels"], d["test_feats"]
train_feats, train_labels = d["train_feats"], d["train_labels"]

small_cnn          selection-set AUC 0.9612
resnet50           selection-set AUC 0.9988
densenet121        selection-set AUC 0.9991
efficientnet_b0    selection-set AUC 0.9984

selected model: densenet121


In [23]:
def fit_temperature(logits, labels, steps=300):
    lg = torch.tensor(logits, dtype=torch.float32).unsqueeze(1)
    lb = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)
    log_t = torch.zeros(1, requires_grad=True)     # T = exp(log_t) stays positive
    crit = nn.BCEWithLogitsLoss()
    opt = torch.optim.LBFGS([log_t], lr=0.05, max_iter=steps)
    def closure():
        opt.zero_grad(); loss = crit(lg / torch.exp(log_t), lb); loss.backward(); return loss
    opt.step(closure)
    return float(torch.exp(log_t).item())

T = fit_temperature(cal_logits, cal_labels)
if not 0.05 <= T <= 20:
    print(f"WARNING: temperature {T:.3f} is outside [0.05, 20]; clipping. Inspect the selected model before using it.")
    T = float(np.clip(T, 0.05, 20))
# Cross-fitting. T and the confidence threshold are both fitted on the calibration set, so the confidences used
# to set tau_c must not come from a T that has seen the same images. Patient-grouped 5-fold cross-fitting:
# for each fold T is fitted on the other four folds and applied to the held-out fold; the pooled out-of-fold
# probabilities set tau_c. The T that is deployed is the one fitted on the whole calibration set. The distance
# score does not depend on T (it uses features, not logits), so tau_d needs no cross-fitting.
from sklearn.model_selection import GroupKFold
cal_pat = splits[splits.split == "cal"].patient_id.to_numpy()
assert len(cal_pat) == len(cal_logits), "calibration order differs between the split table and the saved outputs"
oof_p = np.zeros(len(cal_logits)); fold_T = []
for tr_i, te_i in GroupKFold(n_splits=min(5, len(set(cal_pat)))).split(cal_logits, cal_labels, groups=cal_pat):
    t_k = float(np.clip(fit_temperature(cal_logits[tr_i], cal_labels[tr_i]), 0.05, 20))
    fold_T.append(t_k); oof_p[te_i] = sigmoid(cal_logits[te_i] / t_k)
print("per-fold temperatures:", [round(t, 3) for t in fold_T])
cal_raw, cal_p = sigmoid(cal_logits), sigmoid(cal_logits / T)
test_raw, test_p = sigmoid(test_logits), sigmoid(test_logits / T)
ece = {"cal_before": expected_calibration_error(cal_labels, cal_raw),
       "cal_after": expected_calibration_error(cal_labels, cal_p),
       "test_before": expected_calibration_error(test_labels, test_raw),
       "test_after": expected_calibration_error(test_labels, test_p)}
print(f"temperature {T:.3f}")
print(f"ECE calibration set {ece['cal_before']:.4f} -> {ece['cal_after']:.4f}   "
      f"| test set {ece['test_before']:.4f} -> {ece['test_after']:.4f}  (test not used to fit T)")

def fit_mahalanobis(feats, labels):
    feats = np.asarray(feats, dtype=np.float64)
    means, centred = {}, []
    for c in (0, 1):
        f = feats[labels == c]; mu = f.mean(axis=0)
        means[c] = mu; centred.append(f - mu)
    cov = np.cov(np.concatenate(centred), rowvar=False)
    cov += np.eye(cov.shape[0]) * 1e-3
    return means, np.linalg.pinv(cov)

def mahalanobis(feats, means, prec):
    feats = np.asarray(feats, dtype=np.float64)
    ds = []
    for c in (0, 1):
        diff = feats - means[c]
        ds.append(np.einsum("ij,jk,ik->i", diff, prec, diff))
    return np.min(np.stack(ds, axis=1), axis=1)

def confidence(p):
    return np.where(np.asarray(p) >= 0.5, p, 1 - np.asarray(p))

def fit_thresholds(cal_dist, cal_p, retain_dist=RETAIN_DIST, coverage_conf=COVERAGE_CONF):
    tau_d = float(np.quantile(cal_dist, retain_dist))
    inside = cal_dist <= tau_d
    tau_c = float(np.quantile(confidence(cal_p)[inside], 1 - coverage_conf))
    return tau_d, tau_c

def triage(dist, p, tau_d, tau_c, decision_thr=DECISION_THR):
    """The gate. Identical logic to backend/inference.py."""
    conf = confidence(p)
    return np.where(dist > tau_d, "REJECTED",
           np.where(conf < tau_c, "ESCALATED",
           np.where(np.asarray(p) >= decision_thr, "FLAGGED_URGENT", "CLEARED")))

means, prec = fit_mahalanobis(train_feats, train_labels)
cal_dist = mahalanobis(cal_feats, means, prec)
tau_d, tau_c = fit_thresholds(cal_dist, oof_p)   # out-of-fold confidences, see above
print(f"distance threshold   tau_d = {tau_d:.1f}   (keeps {RETAIN_DIST:.0%} of calibration images)")
print(f"confidence threshold tau_c = {tau_c:.4f}  (keeps {COVERAGE_CONF:.0%} of the in-distribution calibration images)")

np.savez(OUT / "gate_stats.npz", class_means=np.stack([means[0], means[1]]), precision=prec)
gate = {"selected_model": best, "selection_metric": "AUC on the selection set",
        "selection_auc_by_model": sel_auc,
        "temperature": T, "decision_threshold": DECISION_THR,
        "confidence_threshold": tau_c, "distance_threshold": tau_d,
        "retain_dist": RETAIN_DIST, "coverage_conf": COVERAGE_CONF,
        "cross_fitting": {"folds": min(5, len(set(cal_pat))), "grouped_by": "patient", "fold_temperatures": fold_T,
                          "used_for": "confidence threshold (out-of-fold confidences); deployed T is fitted on the whole calibration set"},
        "tie_rule": "accepted when distance <= tau_d and confidence >= tau_c; pneumonia when p >= decision_threshold",
        "ece": ece, "fitted_on": "calibration set (patient-disjoint from train, selection and test)",
        "manifest_sha256": manifest_sha256, "seed": SEED,
        "note": "all parameters fitted on the calibration set, then frozen"}
save_json(gate, "gate_params.json")

per-fold temperatures: [0.783, 0.75, 0.797, 0.733, 0.62]
temperature 0.743
ECE calibration set 0.0044 -> 0.0044   | test set 0.1007 -> 0.1095  (test not used to fit T)
distance threshold   tau_d = 841.7   (keeps 95% of calibration images)
confidence threshold tau_c = 0.9998  (keeps 80% of the in-distribution calibration images)
wrote /kaggle/working/pneumoscan/gate_params.json


In [24]:
# reliability diagrams: calibration set before/after, and TEST set after (T was not fitted on it)
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
panels = [(cal_raw, cal_labels, f"calibration set, before (ECE {ece['cal_before']:.3f})"),
          (cal_p, cal_labels, f"calibration set, after T={T:.2f} (ECE {ece['cal_after']:.3f})"),
          (test_p, test_labels, f"test set, after T={T:.2f} (ECE {ece['test_after']:.3f})")]
for ax, (p, y, title) in zip(axes, panels):
    edges = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (p > lo) & (p <= hi)
        if m.sum():
            xs.append(p[m].mean()); ys.append(y[m].mean())
    ax.plot([0, 1], [0, 1], "k:", lw=0.8)
    ax.plot(xs, ys, "o-", color="#2F4B7C")
    ax.set_title(title, fontsize=9); ax.set_xlabel("predicted probability of pneumonia")
axes[0].set_ylabel("observed frequency of pneumonia")
plt.tight_layout(); plt.savefig(OUT / "calibration.png", dpi=200); plt.show()

## 6. Results: the classifier, then the whole gated pipeline

Everything here is on the **test set** and uses frozen parameters. Intervals are 95%, resampling **patients**, not images.

In [25]:
test_pat = splits[splits.split == "test"].patient_id.to_numpy()
y_test = np.load(OUT / "y_test.npy")
assert (y_test == test_labels).all(), "test order differs between classical and deep pipelines"

def row_for(name, family, y, p):
    base = metrics_at(y, p)
    draws = cluster_bootstrap(test_pat, lambda i: {k: metrics_at(y[i], p[i])[k]
                                                   for k in ("sensitivity", "specificity", "auc")})
    out = {"model": name, "family": family, **base}
    for k in ("sensitivity", "specificity", "auc"):
        out[f"{k}_ci_low"], out[f"{k}_ci_high"] = ci_of(draws, k)
    return out

rows, curves = [], {}
for name, p in classical_probs.items():
    rows.append(row_for(name, "classical", y_test, p)); curves[name] = (y_test, p)
for name in DEEP:
    dd = np.load(OUT / f"outputs_{name}.npz")
    p = sigmoid(dd["test_logits"] / (T if name == best else 1.0))   # only the selected model is calibrated
    rows.append(row_for(name, "deep", dd["test_labels"], p)); curves[name] = (dd["test_labels"], p)

table = pd.DataFrame(rows)
table["params_millions"] = table.model.map(lambda n: training_summary.get(n, {}).get("params_millions"))
table = table.sort_values("auc", ascending=False).reset_index(drop=True)
table.to_csv(OUT / "model_comparison.csv", index=False)
table[["model", "family", "auc", "auc_ci_low", "auc_ci_high", "sensitivity", "sensitivity_ci_low",
       "sensitivity_ci_high", "specificity", "specificity_ci_low", "specificity_ci_high", "f1",
       "params_millions"]].round(3)

,model,family,auc,auc_ci_low,auc_ci_high,sensitivity,sensitivity_ci_low,sensitivity_ci_high,specificity,specificity_ci_low,specificity_ci_high,f1,params_millions
0,resnet50,deep,0.976,0.964,0.986,0.990,0.977,0.998,0.714,0.653,0.772,0.916,23.51
1,efficientnet_b0,deep,0.971,0.958,0.982,0.992,0.982,1.000,0.749,0.688,0.806,0.926,4.01
2,densenet121,deep,0.971,0.957,0.982,0.997,0.991,1.000,0.654,0.588,0.717,0.905,6.95
3,random_forest,classical,0.949,0.932,0.964,0.992,0.982,1.000,0.377,0.308,0.440,0.839,NaN
4,svm_rbf,classical,0.940,0.921,0.959,0.995,0.987,1.000,0.433,0.366,0.502,0.853,NaN
5,logistic_regression,classical,0.938,0.918,0.956,0.992,0.983,1.000,0.476,0.410,0.540,0.861,NaN
6,small_cnn,deep,0.910,0.886,0.932,0.904,0.874,0.934,0.745,0.686,0.795,0.879,0.24


In [26]:
plt.figure(figsize=(6, 5))
for name, (yy, p) in curves.items():
    fpr, tpr, _ = roc_curve(yy, p)
    plt.plot(fpr, tpr, lw=1.6, label=f"{name} ({roc_auc_score(yy, p):.3f})")
plt.plot([0, 1], [0, 1], "k:", lw=0.8)
plt.xlabel("1 − specificity"); plt.ylabel("sensitivity")
plt.title("ROC curves, held-out test set"); plt.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "roc_curves.png", dpi=200); plt.show()

### The gated pipeline

For the selected model, every test image goes through the frozen gate. We report:

1. **Retained coverage** (autonomy rate): the share of images that receive CLEARED or FLAGGED_URGENT
2. **What happened to each class**: counts per final state
3. **Errors on the images the gate accepted**, per class, with Wilson intervals
4. **False-clearance rate**: among CLEARED images, the share that actually had pneumonia
5. **Ungated vs gated**, same images, same decision threshold, patient-level bootstrap intervals

In [27]:
test_dist = mahalanobis(test_feats, means, prec)
state = triage(test_dist, test_p, tau_d, tau_c)
res = splits[splits.split == "test"][["file", "patient_id", "label"]].reset_index(drop=True)
res["probability"], res["confidence"], res["distance"], res["state"] = test_p, confidence(test_p), test_dist, state
res.to_csv(OUT / "gate_decisions_test.csv", index=False)

accepted = np.isin(state, ["CLEARED", "FLAGGED_URGENT"])
n = len(res)

by_class = pd.crosstab(res.label.map({0: "normal", 1: "pneumonia"}), res.state)
for s in ["CLEARED", "FLAGGED_URGENT", "ESCALATED", "REJECTED"]:
    if s not in by_class: by_class[s] = 0
by_class = by_class[["CLEARED", "FLAGGED_URGENT", "ESCALATED", "REJECTED"]]
by_class["n"] = by_class.sum(axis=1)
by_class["coverage"] = (by_class.CLEARED + by_class.FLAGGED_URGENT) / by_class.n
by_class.to_csv(OUT / "gating_by_class.csv")
print(by_class, "\n")

fn = int(((res.label == 1) & (res.state == "CLEARED")).sum())
fp = int(((res.label == 0) & (res.state == "FLAGGED_URGENT")).sum())
n_cleared, n_flagged = int((res.state == "CLEARED").sum()), int((res.state == "FLAGGED_URGENT").sum())
acc_pneu, acc_norm = int(((res.label == 1) & accepted).sum()), int(((res.label == 0) & accepted).sum())

errors = {
    "false_negatives_among_accepted_pneumonia": {"k": fn, "n": acc_pneu, "rate": fn / acc_pneu if acc_pneu else None, "wilson95": wilson(fn, acc_pneu)},
    "false_positives_among_accepted_normal": {"k": fp, "n": acc_norm, "rate": fp / acc_norm if acc_norm else None, "wilson95": wilson(fp, acc_norm)},
    "false_clearance_rate_among_CLEARED": {"k": fn, "n": n_cleared, "rate": fn / n_cleared if n_cleared else None, "wilson95": wilson(fn, n_cleared)},
    "false_alarm_rate_among_FLAGGED": {"k": fp, "n": n_flagged, "rate": fp / n_flagged if n_flagged else None, "wilson95": wilson(fp, n_flagged)},
}
for k, v in errors.items():
    r = f"{v['rate']:.3f}" if v["rate"] is not None else "n/a"
    print(f"{k:45s} {v['k']}/{v['n']} = {r}   95% CI {v['wilson95'][0]:.3f}–{v['wilson95'][1]:.3f}")

state      CLEARED  FLAGGED_URGENT  ESCALATED  REJECTED    n  coverage
label                                                                 
normal          36              14        154        27  231  0.216450
pneumonia        0             325         47        15  387  0.839793 

false_negatives_among_accepted_pneumonia      0/325 = 0.000   95% CI 0.000–0.012
false_positives_among_accepted_normal         14/50 = 0.280   95% CI 0.175–0.417
false_clearance_rate_among_CLEARED            0/36 = 0.000   95% CI 0.000–0.096
false_alarm_rate_among_FLAGGED                14/339 = 0.041   95% CI 0.025–0.068


In [28]:
y = res.label.to_numpy(); pr = res.probability.to_numpy(); acc = accepted

def gate_stats(i):
    yi, pi, ai = y[i], pr[i], acc[i]
    g = metrics_at(yi[ai], pi[ai]) if ai.sum() else {}
    u = metrics_at(yi, pi)
    return {"coverage": ai.mean(),
            "ungated_sensitivity": u["sensitivity"], "ungated_specificity": u["specificity"],
            "gated_sensitivity": g.get("sensitivity", np.nan), "gated_specificity": g.get("specificity", np.nan)}

point = gate_stats(np.arange(n))
draws = cluster_bootstrap(res.patient_id.to_numpy(), gate_stats)
summary = {}
for k, v in point.items():
    lo, hi = ci_of(draws, k)
    summary[k] = {"value": float(v), "ci95": [lo, hi]}
    print(f"{k:22s} {v:.3f}   95% CI {lo:.3f}–{hi:.3f}")

gating_results = {
    "selected_model": best, "n_test": n,
    "definitions": {"decision_threshold": DECISION_THR, "confidence_threshold": tau_c, "distance_threshold": tau_d,
                    "temperature": T, "retain_dist": RETAIN_DIST, "coverage_conf": COVERAGE_CONF,
                    "fitted_on": "calibration set only"},
    "final_states": {s: int((res.state == s).sum()) for s in ["CLEARED", "FLAGGED_URGENT", "ESCALATED", "REJECTED"]},
    "retained_coverage": summary["coverage"],
    "ungated_all_images": {"sensitivity": summary["ungated_sensitivity"], "specificity": summary["ungated_specificity"]},
    "gated_accepted_only": {"sensitivity": summary["gated_sensitivity"], "specificity": summary["gated_specificity"]},
    "errors_on_accepted": errors,
    "ece": ece,
}
save_json(gating_results, "gating_results.json")

res[(res.label == 1) & (res.state == "CLEARED")].to_csv(OUT / "accepted_false_negatives.csv", index=False)
print(f"\naccepted false negatives (pneumonia that was CLEARED): {fn} -> accepted_false_negatives.csv")

coverage               0.607   95% CI 0.555–0.654
ungated_sensitivity    0.997   95% CI 0.991–1.000
ungated_specificity    0.654   95% CI 0.588–0.717
gated_sensitivity      1.000   95% CI 1.000–1.000
gated_specificity      0.720   95% CI 0.592–0.841
wrote /kaggle/working/pneumoscan/gating_results.json

accepted false negatives (pneumonia that was CLEARED): 0 -> accepted_false_negatives.csv


In [29]:
# Coverage–risk table. Thresholds for each coverage target are fitted on the calibration set, never on test.
sweep = []
for cov in (1.00, 0.90, 0.80, 0.70, 0.60, 0.50):
    td, tc = fit_thresholds(cal_dist, oof_p, RETAIN_DIST, cov)   # out-of-fold confidences, as for the operating point
    st = triage(test_dist, test_p, td, tc)
    ok = np.isin(st, ["CLEARED", "FLAGGED_URGENT"])
    g = metrics_at(y[ok], pr[ok]) if ok.sum() else {}
    sweep.append({"target_coverage_on_calibration": cov, "tau_c": tc, "test_coverage": float(ok.mean()),
                  "accepted_sensitivity": g.get("sensitivity"), "accepted_specificity": g.get("specificity"),
                  "false_negatives_accepted": g.get("fn"), "false_positives_accepted": g.get("fp")})
sweep = pd.DataFrame(sweep)
sweep.to_csv(OUT / "coverage_risk.csv", index=False)
sweep.round(3)

,target_coverage_on_calibration,tau_c,test_coverage,accepted_sensitivity,accepted_specificity,false_negatives_accepted,false_positives_accepted
0,1.0,0.501,0.932,0.997,0.623,1,77
1,0.9,0.996,0.720,1.000,0.652,0,32
2,0.8,1.000,0.607,1.000,0.720,0,14
3,0.7,1.000,0.539,1.000,0.744,0,10
4,0.6,1.000,0.468,1.000,0.792,0,5
5,0.5,1.000,0.369,1.000,0.917,0,1


In [30]:
cm_u = metrics_at(y, pr)
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.8))
for ax, (m, title) in zip(axes, [(cm_u, f"ungated (n={n})"), (metrics_at(y[acc], pr[acc]), f"accepted by the gate (n={int(acc.sum())})")]):
    cm = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14)
    ax.set_xticks([0, 1], ["pred normal", "pred pneumonia"]); ax.set_yticks([0, 1], ["actual normal", "actual pneumonia"])
    ax.set_title(title, fontsize=10)
plt.suptitle(f"{best}: test set", fontsize=11)
plt.tight_layout(); plt.savefig(OUT / "confusion_best.png", dpi=200); plt.show()

## 7. Out-of-distribution behaviour of the frozen gate

Three kinds of input the gate should refuse or escalate, pushed through the **same frozen parameters**:

1. **Degraded test images** generated here: blur, under-exposure, over-exposure, rotation by 90°, 40% centre crop, heavy noise
2. **Non-radiographic images**: two photographs bundled with scikit-learn, plus synthetic noise, blank and gradient images. Small, so read as examples, not a benchmark
3. **External folders** (optional): set `OOD_ROOT` to a folder with one sub-folder per category, e.g. `adult_rsna/`

"Refused" means REJECTED or ESCALATED. Giving CLEARED or FLAGGED_URGENT to one of these inputs is a **failure of the gate**.

In [31]:
model, _, _ = build(best)
model.load_state_dict(torch.load(OUT / f"{best}.pt", map_location=DEV)); model.to(DEV).eval()

@torch.no_grad()
def score_pils(pils):
    ps, ds = [], []
    for i in range(0, len(pils), BATCH):
        x = torch.stack([eval_tf(im.convert("RGB")) for im in pils[i:i + BATCH]]).to(DEV)
        lg = model(x).squeeze(1).cpu().numpy().reshape(-1)
        ft = embed(model, best, x).cpu().numpy()
        ps.append(sigmoid(lg / T)); ds.append(mahalanobis(ft, means, prec))
    p, dd = np.concatenate(ps), np.concatenate(ds)
    return triage(dd, p, tau_d, tau_c)

def degrade(im, kind, rng):
    im = im.convert("L").convert("RGB")
    if kind == "blur":      return im.filter(ImageFilter.GaussianBlur(radius=max(3, im.width // 60)))
    if kind == "dark":      return ImageEnhance.Brightness(im).enhance(0.25)
    if kind == "bright":    return ImageEnhance.Brightness(im).enhance(2.8)
    if kind == "rot90":     return im.rotate(90, expand=True)
    if kind == "crop40":
        w, h = im.size; cw, ch = int(w * 0.4), int(h * 0.4)
        return im.crop(((w - cw) // 2, (h - ch) // 2, (w + cw) // 2, (h + ch) // 2)).resize((w, h))
    if kind == "noise":
        a = np.asarray(im, dtype=np.float32) + rng.normal(0, 60, (im.height, im.width, 3))
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    raise ValueError(kind)

rng = np.random.default_rng(SEED)
test_files = splits[splits.split == "test"].path.sample(min(100, int((splits.split == "test").sum())), random_state=SEED).tolist()
categories = {}
for kind in ("blur", "dark", "bright", "rot90", "crop40", "noise"):
    categories[f"degraded_{kind}"] = [degrade(Image.open(p), kind, rng) for p in test_files]

nonchest = []
try:
    from sklearn.datasets import load_sample_images
    nonchest += [Image.fromarray(a) for a in load_sample_images().images]
except Exception as e:
    print("sklearn sample images unavailable:", e)
side = 256
nonchest += [Image.fromarray(rng.integers(0, 256, (side, side, 3), dtype=np.uint8)),
             Image.new("RGB", (side, side), (0, 0, 0)), Image.new("RGB", (side, side), (255, 255, 255)),
             Image.fromarray(np.tile(np.linspace(0, 255, side, dtype=np.uint8), (side, 1))).convert("RGB")]
categories["non_radiographic"] = nonchest

ood_root = os.environ.get("OOD_ROOT")
if ood_root and Path(ood_root).exists():
    for sub in sorted(p for p in Path(ood_root).iterdir() if p.is_dir()):
        imgs = [q for q in sub.rglob("*") if q.suffix.lower() in {".jpg", ".jpeg", ".png"}][:200]
        if imgs:
            categories[f"external_{sub.name}"] = [Image.open(q) for q in imgs]

ood_rows = []
for cat, pils in categories.items():
    st = score_pils(pils)
    k_rej, k_ref = int((st == "REJECTED").sum()), int(np.isin(st, ["REJECTED", "ESCALATED"]).sum())
    ood_rows.append({"category": cat, "n": len(st), "rejected": k_rej, "refused": k_ref,
                     "rejected_rate": k_rej / len(st), "refused_rate": k_ref / len(st),
                     "refused_wilson_low": wilson(k_ref, len(st))[0], "refused_wilson_high": wilson(k_ref, len(st))[1],
                     "wrongly_decided": len(st) - k_ref})
ood = pd.DataFrame(ood_rows)
ood.to_csv(OUT / "ood_results.csv", index=False)
print("in-distribution reference: share of test images the gate REJECTED =",
      f"{(res.state == 'REJECTED').mean():.3f}")
ood.round(3)

in-distribution reference: share of test images the gate REJECTED = 0.068


,category,n,rejected,refused,rejected_rate,refused_rate,refused_wilson_low,refused_wilson_high,wrongly_decided
0,degraded_blur,100,100,100,1.0,1.0,0.963,1.0,0
1,degraded_dark,100,100,100,1.0,1.0,0.963,1.0,0
2,degraded_bright,100,100,100,1.0,1.0,0.963,1.0,0
3,degraded_rot90,100,100,100,1.0,1.0,0.963,1.0,0
4,degraded_crop40,100,100,100,1.0,1.0,0.963,1.0,0
5,degraded_noise,100,100,100,1.0,1.0,0.963,1.0,0
6,non_radiographic,6,6,6,1.0,1.0,0.610,1.0,0


## 8. Grad-CAM

Checks the model looks at lung fields rather than borders, text markers or scanner artefacts —
the shortcut learning Zech et al. (2018) documented.

In [32]:
if best == "small_cnn" or SMOKE:
    print("Grad-CAM set up for the pretrained backbones; skipped for small_cnn / smoke test.")
else:
    target_layer = {"resnet50": lambda m: m.layer4[-1],
                    "densenet121": lambda m: m.features[-1],
                    "efficientnet_b0": lambda m: m.features[-1]}[best](model)

    store = {}
    def _fwd_hook(m, i, o):
        # torchvision's DenseNet applies an in-place ReLU to this output, which breaks full backward hooks.
        # Return a clone and attach a tensor hook to it instead.
        a = o.clone()
        a.register_hook(lambda g: store.__setitem__("grad", g))
        act = a.detach().clone()
        store["act"] = F.relu(act) if best == "densenet121" else act
        return a
    _hook = target_layer.register_forward_hook(_fwd_hook)
    model.eval()

    sample = splits[splits.split == "test"].groupby("label").head(3)
    fig, axes = plt.subplots(2, 3, figsize=(11, 7))
    for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
        with Image.open(row.path) as im:
            rgb = im.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
        x = eval_tf(rgb).unsqueeze(0).to(DEV)
        with torch.enable_grad():
            out = model(x); model.zero_grad(); out.sum().backward()
        wts = store["grad"].mean(dim=(2, 3), keepdim=True)
        cam = F.relu((wts * store["act"]).sum(1, keepdim=True))
        cam = F.interpolate(cam, (IMG_SIZE, IMG_SIZE), mode="bilinear",
                            align_corners=False)[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        p = float(torch.sigmoid(out.detach() / T))
        ax.imshow(np.asarray(rgb)); ax.imshow(cam, cmap="jet", alpha=0.4)
        ax.set_title(f"{'pneumonia' if row.label else 'normal'} — p={p:.2f}", fontsize=10)
        ax.axis("off")
    _hook.remove()
    plt.tight_layout(); plt.savefig(OUT / "gradcam_samples.png", dpi=200); plt.show()

## 9. Component comparison at matched coverage

In [35]:
cal_conf = confidence(oof_p)                 
test_conf = confidence(test_p)
inside_frac = float((cal_dist <= tau_d).mean())
y_t = res.label.to_numpy(); p_t = res.probability.to_numpy(); pid_t = res.patient_id.to_numpy()
pred_t = (p_t >= DECISION_THR).astype(int)

def accept_mask(system, cov):
    if system == "ungated":
        return np.ones(len(y_t), bool)
    if system == "calibration_only":         # confidence rule only, no distance check
        return test_conf >= float(np.quantile(cal_conf, 1 - cov))
    if system == "ood_only":                 # distance rule only, no confidence check
        return test_dist <= float(np.quantile(cal_dist, cov))
    td, tc = fit_thresholds(cal_dist, oof_p, RETAIN_DIST, min(1.0, cov / inside_frac))   # combined
    return (test_dist <= td) & (test_conf >= tc)

def acc_stats(i, ok):
    yi, pi_, oki = y_t[i], pred_t[i], ok[i]
    tp, tn, fp, fn = confusion(yi[oki], pi_[oki])
    return {"coverage": float(oki.mean()),
            "sensitivity": tp / (tp + fn) if tp + fn else np.nan,
            "specificity": tn / (tn + fp) if tn + fp else np.nan}

op_cov = round(inside_frac * COVERAGE_CONF, 3)          # coverage of the deployed operating point
levels = sorted({0.95, 0.90, 0.80, 0.70, 0.60, op_cov}, reverse=True)
plan = [("ungated", 1.0)] + [(s, c) for c in levels for s in ("calibration_only", "ood_only", "combined")]
pneu, norm = y_t == 1, y_t == 0
rows = []
for system, cov in plan:
    ok = accept_mask(system, cov)
    tp, tn, fp, fn = confusion(y_t[ok], pred_t[ok])
    draws = cluster_bootstrap(pid_t, lambda i: acc_stats(i, ok), n_boot=500)
    r = {"system": system, "target_coverage": cov, "test_coverage": float(ok.mean()),
         "accepted_pneumonia": int((ok & pneu).sum()), "accepted_normal": int((ok & norm).sum()),
         "missed_pneumonia_accepted": fn, "false_alarms_accepted": fp,
         "refused_pneumonia": int((~ok & pneu).sum()), "refused_normal": int((~ok & norm).sum()),
         "accepted_sensitivity": tp / (tp + fn) if tp + fn else np.nan,
         "accepted_specificity": tn / (tn + fp) if tn + fp else np.nan}
    for k in ("coverage", "sensitivity", "specificity"):
        r[f"{k}_ci_lo"], r[f"{k}_ci_hi"] = ci_of(draws, k)
    rows.append(r)
comp = pd.DataFrame(rows)
comp.to_csv(OUT / "component_comparison.csv", index=False)
print(f"operating point of the deployed gate = coverage {op_cov}")
comp.round(3)

# Exact matching, for comparison only. Here each single-component rule is given the SAME test-set coverage as
# the deployed gate. The threshold therefore uses the test set's score ranking, so this is a descriptive
# comparison, not a deployable rule. Report it as such, next to the calibration-fitted table above.
k = int(round(accept_mask("combined", op_cov).mean() * len(y_t)))
ok_by_system = {
    "calibration_only": test_conf >= np.sort(test_conf)[::-1][k - 1],
    "ood_only": test_dist <= np.sort(test_dist)[k - 1],
    "combined": accept_mask("combined", op_cov),
}
rows_m = []
for system, ok in ok_by_system.items():
    tp, tn, fp, fn = confusion(y_t[ok], pred_t[ok])
    draws = cluster_bootstrap(pid_t, lambda i: acc_stats(i, ok), n_boot=500)
    r = {"system": system, "test_coverage": float(ok.mean()),
         "accepted_pneumonia": int((ok & pneu).sum()), "accepted_normal": int((ok & norm).sum()),
         "missed_pneumonia_accepted": fn, "false_alarms_accepted": fp,
         "refused_pneumonia": int((~ok & pneu).sum()), "refused_normal": int((~ok & norm).sum()),
         "accepted_sensitivity": tp / (tp + fn) if tp + fn else np.nan,
         "accepted_specificity": tn / (tn + fp) if tn + fp else np.nan}
    for key in ("sensitivity", "specificity"):
        r[f"{key}_ci_lo"], r[f"{key}_ci_hi"] = ci_of(draws, key)
    rows_m.append(r)
comp_matched = pd.DataFrame(rows_m)
comp_matched.to_csv(OUT / "component_comparison_matched.csv", index=False)
comp_matched.round(3)

operating point of the deployed gate = coverage 0.76


,system,test_coverage,accepted_pneumonia,accepted_normal,missed_pneumonia_accepted,false_alarms_accepted,refused_pneumonia,refused_normal,accepted_sensitivity,accepted_specificity,sensitivity_ci_lo,sensitivity_ci_hi,specificity_ci_lo,specificity_ci_hi
0,calibration_only,0.607,319,56,0,14,68,175,1.000,0.75,1.000,1.0,0.622,0.858
1,ood_only,0.607,251,124,1,62,136,107,0.996,0.50,0.986,1.0,0.405,0.588
2,combined,0.607,325,50,0,14,62,181,1.000,0.72,1.000,1.0,0.590,0.833
